# Mamba Paper Study: Single Colab Workflow

This notebook runs the original repository workflow in one linear Colab session:

1. environment and dependency setup
2. Pile-compatible data streaming and GPT-NeoX tokenization
3. Mamba-130M training
4. GPT-2-style Transformer baseline training
5. held-out perplexity and text generation
6. throughput and peak-memory benchmarking
7. plot generation for presentation artifacts

The workflow keeps the original scripts as helper modules and only reduces compute scale for Colab GPUs. The dataset remains The Pile family via `monology/pile-uncopyrighted`.

## 1. Runtime Check

Use a Colab GPU runtime. T4 and L4 are supported for demonstration scale; A100 remains closer to the original project setup.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

try:
    import torch
    print("Python:", sys.version)
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA:", torch.version.cuda)
        print("BF16 supported:", torch.cuda.is_bf16_supported())
except ImportError:
    print("PyTorch is not installed. Colab GPU runtimes normally include CUDA PyTorch.")

## 2. Get The Repository

If this notebook is opened directly from GitHub in Colab, clone the repo. If you are already inside the repo, this cell leaves the working tree alone.

In [ ]:
REPO_URL = "https://github.com/seriouslysahid/genai-mamba-assignment"
REPO_DIR = Path("/content/genai-mamba-assignment")

if not Path("scripts/train_mamba.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

print("Working directory:", Path.cwd())
print("Scripts:", sorted(p.name for p in Path("scripts").glob("*.py")))

## 3. Install Dependencies

This notebook assumes Colab Runtime Version `2025.07` for the Mamba CUDA stack:

- Python 3.11.x
- PyTorch 2.6.x
- CUDA 12.x

The dependency cell installs official prebuilt GitHub release wheels for `causal-conv1d` and `mamba-ssm`. It chooses the correct CXX11 ABI wheel from `torch._C._GLIBCXX_USE_CXX11_ABI`, uninstalls stale wheels first, and runs a small Mamba model-construction smoke test before training.


In [ ]:
import subprocess
import sys
from urllib.parse import quote
from tqdm.auto import tqdm

# This install path assumes Colab Runtime Version 2025.07:
# Python 3.11.x + PyTorch 2.6.x + CUDA 12.x.
# It installs official prebuilt GitHub release wheels for mamba-ssm and causal-conv1d.
REINSTALL_MAMBA_DEPS = True
MAMBA_VERSION = "2.2.3.post2"
MAMBA_TAG = "v2.2.3.post2"
CAUSAL_CONV1D_VERSION = "1.5.0.post5"
CAUSAL_CONV1D_TAG = "v1.5.0.post5"


def run_step(label, cmd, check=True, quiet=False):
    print(f"\n[{label}] {' '.join(cmd)}")
    if quiet:
        return subprocess.run(cmd, check=check)
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


def assert_2025_07_runtime(torch):
    py = sys.version_info
    torch_version = torch.__version__.split("+")[0]
    cuda_version = torch.version.cuda or ""
    problems = []
    if (py.major, py.minor) != (3, 11):
        problems.append(f"Python must be 3.11.x, got {py.major}.{py.minor}.{py.micro}")
    if not torch_version.startswith("2.6."):
        problems.append(f"PyTorch must be 2.6.x, got {torch.__version__}")
    if not cuda_version.startswith("12."):
        problems.append(f"Torch CUDA must be 12.x, got {cuda_version or 'none'}")
    if problems:
        raise RuntimeError(
            "This notebook install cell is configured for Colab Runtime Version 2025.07.\n"
            + "\n".join(f"- {p}" for p in problems)
            + "\nSelect Runtime -> Change runtime type -> Runtime version 2025.07, then restart."
        )


def github_wheel_url(repo, tag, filename):
    return f"https://github.com/{repo}/releases/download/{tag}/{quote(filename)}"


def prebuilt_wheel_urls(torch):
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    cuda_tag = "cu12"
    torch_tag = "torch2.6"
    platform = "linux_x86_64"
    causal = f"causal_conv1d-{CAUSAL_CONV1D_VERSION}+{cuda_tag}{torch_tag}cxx11abi{abi}-{py_tag}-{py_tag}-{platform}.whl"
    mamba = f"mamba_ssm-{MAMBA_VERSION}+{cuda_tag}{torch_tag}cxx11abi{abi}-{py_tag}-{py_tag}-{platform}.whl"
    return [
        github_wheel_url("Dao-AILab/causal-conv1d", CAUSAL_CONV1D_TAG, causal),
        github_wheel_url("state-spaces/mamba", MAMBA_TAG, mamba),
    ]

install_steps = [
    "project deps",
    "runtime check",
    "prebuilt mamba wheels",
    "import smoke test",
]

with tqdm(total=len(install_steps), desc="dependency setup", unit="step") as pbar:
    run_step("project deps", [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], quiet=True)
    run_step("wheel runtime deps", [sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "wheel"], quiet=True)
    pbar.update(1)

    import torch
    print("Python:", sys.version)
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CXX11 ABI:", torch._C._GLIBCXX_USE_CXX11_ABI)
    assert_2025_07_runtime(torch)
    pbar.update(1)

    if REINSTALL_MAMBA_DEPS:
        run_step("remove old mamba deps", [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "mamba-ssm", "causal-conv1d"], check=False, quiet=True)

    wheel_urls = prebuilt_wheel_urls(torch)
    print("Selected wheels:")
    for url in wheel_urls:
        print(" -", url.rsplit("/", 1)[-1])
    run_step("prebuilt mamba wheels", [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", *wheel_urls])
    pbar.update(1)

    import causal_conv1d
    import mamba_ssm
    from mamba_ssm.models.config_mamba import MambaConfig
    from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    smoke = MambaLMHeadModel(MambaConfig(d_model=64, n_layer=2, vocab_size=128)).to(device)
    del smoke
    print("Mamba smoke test passed.")
    pbar.update(1)

subprocess.run([sys.executable, "scripts/check_env.py"], check=False)

## 4. Colab Scale Configuration

The model definitions remain the original Mamba-130M and comparable GPT-2-style Transformer baseline. The defaults below only reduce examples, sequence length, batch size, steps, and benchmark sweep for Colab runtime and VRAM.

In [ ]:
from pathlib import Path
import sys
import torch

sys.path.insert(0, str(Path("scripts").resolve()))

DATASET_NAME = "monology/pile-uncopyrighted"
TOKENIZER_NAME = "EleutherAI/gpt-neox-20b"

# Balanced Colab demo shard. Your 5k/500 run produced ~6.7M train tokens;
# this default should produce roughly ~33M train tokens while staying small on disk.
NUM_TRAIN_EXAMPLES = 25_000
NUM_VAL_EXAMPLES = 2_000

# Larger options if the runtime is stable:
#   Colab Pro/L4:      50_000 train, 5_000 val
#   Stretch/A100:     100_000 train, 10_000 val
# Full monology/pile-uncopyrighted is ~335 GB raw / ~176M rows and is not Colab-practical.
SEQ_LEN = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_STEPS = 100
EVAL_INTERVAL = 50
EVAL_STEPS = 10
LOG_INTERVAL = 10

def choose_dtype():
    if not torch.cuda.is_available():
        return "float32"
    major, minor = torch.cuda.get_device_capability(0)
    # BF16 kernels require Ampere (sm_80) or newer. T4 is sm_75, so use FP16.
    if major >= 8 and torch.cuda.is_bf16_supported():
        return "bfloat16"
    return "float16"


DTYPE = choose_dtype()
BENCH_SEQ_LENS = [256, 512, 1024, 2048, 4096]
BENCH_BATCH_SIZE = 1

Path("data").mkdir(exist_ok=True)
Path("out").mkdir(exist_ok=True)
print({
    "dataset": DATASET_NAME,
    "num_train_examples": NUM_TRAIN_EXAMPLES,
    "num_val_examples": NUM_VAL_EXAMPLES,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "max_steps": MAX_STEPS,
    "dtype": DTYPE,
    "gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
})

## 5. Download And Tokenize Data

This preserves the original streaming/tokenization flow: stream The Pile-family text, tokenize with the GPT-NeoX tokenizer, and write `data/train.bin` and `data/val.bin` as uint16 token shards.

In [ ]:
cmd = [
    sys.executable, "scripts/prepare_data.py",
    "--dataset_name", DATASET_NAME,
    "--tokenizer_name", TOKENIZER_NAME,
    "--num_train", str(NUM_TRAIN_EXAMPLES),
    "--num_val", str(NUM_VAL_EXAMPLES),
    "--out_dir", "data",
]
subprocess.run(cmd, check=True)

for path in [Path("data/train.bin"), Path("data/val.bin")]:
    print(path, f"{path.stat().st_size / 1e6:.2f} MB")

## 6. Train Mamba-130M

The helper uses the original Mamba-1-style 24 layer, d_model 768, d_state 16 configuration.

In [ ]:
from types import SimpleNamespace
from train_mamba import train

train(SimpleNamespace(
    model="mamba",
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    seq_len=SEQ_LEN,
    dtype=DTYPE,
    eval_interval=EVAL_INTERVAL,
    eval_steps=EVAL_STEPS,
    log_interval=LOG_INTERVAL,
    warmup_steps=min(20, MAX_STEPS),
    data_dir="data",
    out_dir="out",
))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Train Transformer Baseline

The baseline remains a GPT-2-style Transformer with comparable hidden size and parameter scale.

In [ ]:
train(SimpleNamespace(
    model="transformer",
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    seq_len=SEQ_LEN,
    dtype=DTYPE,
    eval_interval=EVAL_INTERVAL,
    eval_steps=EVAL_STEPS,
    log_interval=LOG_INTERVAL,
    warmup_steps=min(20, MAX_STEPS),
    data_dir="data",
    out_dir="out",
))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. Evaluate Checkpoints

Evaluation computes validation negative log likelihood/perplexity and produces text samples from fixed prompts.

In [ ]:
from evaluate_mamba import main as evaluate_main

for model_name in ["mamba", "transformer"]:
    evaluate_main(SimpleNamespace(
        model=model_name,
        ckpt=f"out/{model_name}_final.pt",
        batch_size=BATCH_SIZE,
        max_batches=25,
        data_dir="data",
        out_dir="out",
        seq_len=SEQ_LEN,
        dtype=DTYPE,
    ))
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Benchmark Throughput And Memory

This preserves the original benchmark intent: same random token inputs, same model families, sequence-length sweep, tokens/sec, and peak allocated GPU memory. OOMs are recorded instead of stopping the notebook.

In [ ]:
from benchmark_mamba import run as benchmark_run

benchmark_run(SimpleNamespace(
    seq_lens=BENCH_SEQ_LENS,
    batch_size=BENCH_BATCH_SIZE,
    dtype=DTYPE,
    out_dir="out",
))

## 10. Plot Results

This generates the same presentation-oriented artifacts as the original script: training loss, validation perplexity, throughput, memory, and speedup ratio.

In [ ]:
from plot_results import main as plot_main

plot_main(SimpleNamespace(log_dir="out", out_dir="out"))
print("Output files:")
for path in sorted(Path("out").glob("*")):
    print(" -", path)

## 11. Display Key Outputs

The generated PNGs and JSON files are saved in `out/`. Use these in the academic presentation or download them from the Colab file browser.

In [ ]:
from IPython.display import display, Image

for name in ["training_loss.png", "val_perplexity.png", "throughput.png", "memory.png", "speedup.png"]:
    path = Path("out") / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))